# 🚀 Bane Agent: Live GPU Inference Server & Public Tunnel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariveshkoshta-spec/Bane_Agent/blob/main/Bane_Live_GPU_Server.ipynb)

This notebook hosts your fine-tuned **DPO LLaMA-3-8B Text-to-SQL Model** on a free NVIDIA T4 GPU and opens a zero-config public HTTPS tunnel (Cloudflare Tunnel).

You can then run the **Bane CLI on your local Mac** to send queries directly to this GPU server, execute them against your local SQLite database, and display rich results!

### Step 1: Verify T4 GPU & Fast Library Check

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ GPU NOT ACTIVE! In top menu: Runtime -> Change runtime type -> select 'T4 GPU' -> Save"
print(f"✅ Active GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB VRAM)")

try:
    import peft
    import bitsandbytes
    import faiss
    import sentence_transformers
    import uvicorn
    import fastapi
    print("⚡ All core libraries ready in 0.01s!")
except ImportError:
    print("⏳ Installing required libraries (~15s)...")
    !pip install -q peft bitsandbytes faiss-cpu sentence-transformers uvicorn fastapi rich
    print("✅ Setup complete!")

### Step 2: Fresh Code Sync (Forced Latest Git Commit)

In [ ]:
import os
if not os.path.exists("/content/Bane_Agent") and not os.path.exists("Bane_Agent"):
    !git clone https://github.com/pariveshkoshta-spec/Bane_Agent.git
    %cd Bane_Agent
else:
    if os.path.exists("/content/Bane_Agent"):
        %cd /content/Bane_Agent
    !git fetch origin main
    !git reset --hard origin/main

print("✅ Repository synchronized with latest commit on main.")

### Step 3: Mount Google Drive & Extract Fine-Tuned DPO Adapters

In [ ]:
import os
from google.colab import drive

# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

target_adapter_dir = "results/bane_dpo_lora_adapters"
os.makedirs(target_adapter_dir, exist_ok=True)

if os.path.exists('/content/drive/MyDrive/bane_dpo_lora_adapters.zip'):
    print("📦 Extracting LoRA adapters from Google Drive...")
    !unzip -q -o /content/drive/MyDrive/bane_dpo_lora_adapters.zip -d results/bane_dpo_lora_adapters
    print("✅ LoRA adapters ready in results/bane_dpo_lora_adapters!")
elif os.path.exists('bane_dpo_lora_adapters.zip'):
    print("📦 Extracting LoRA adapters from repo archive...")
    !unzip -q -o bane_dpo_lora_adapters.zip -d results/bane_dpo_lora_adapters
    print("✅ LoRA adapters ready in results/bane_dpo_lora_adapters!")
else:
    print("⚠️ Please verify that bane_dpo_lora_adapters.zip is in your Drive or repo root.")

### Step 4: Launch Live GPU Server with Cloudflare Public Tunnel

Run this cell to start the server! It will display your live **Public API URL**.
Keep this cell running while interacting from your Mac terminal.

In [ ]:
!python scripts/launch_cloud_api.py